In [ ]:
#| default_exp handlers.pipeline.gates

# Gates

Static Gate 1 quarantine, deep Gate 2 validation, and the human-facing remediation scaffolds for boundary and callback fixes.

In [ ]:
#| export
from __future__ import annotations
import sys
from pathlib import Path
from typing import Any, Optional
import pandas as pd
import yaml
from pydantic import BaseModel, ValidationError
from marisco.configs import NC_GLOBAL_ATTRS
from marisco.handlers.pipeline.contracts import HandlerConfig, _RawHandlerContract, _MARIS_REQUIRED

## Gate Helpers

In [ ]:
#| export
_DEEP_CRITICAL = frozenset({"LAT", "LON", "TIME"})
_DEFAULT_LOADER_PATHS = frozenset({
    "marisco.handlers.pipeline.loader.load_data",
    "marisco.handlers.pipeline.loader:load_data",
})


def _yaml_step1_message(err: ValidationError) -> str:
    details = []
    for item in err.errors():
        loc = ".".join(str(p) for p in item["loc"])
        details.append(f"- {loc}: {item['msg']}")
    body = "\n".join(details) or "- YAML contract could not be validated."
    return (
        "Step 1: Type mismatch or missing fields detected in your YAML contract.\n"
        "Please fix the YAML configuration first; runtime callback guidance is withheld.\n"
        f"{body}"
    )


def _unknown_global_attrs_message(keys: set[str]) -> str:
    joined = ", ".join(sorted(keys))
    return (
        "Step 1: Unknown NetCDF global attribute(s) detected in output.global_attrs.\n"
        "Only template-supported names are allowed here.\n"
        f"- output.global_attrs: {joined}"
    )


class _GapDiagnostics(BaseModel):
    title: str = ""
    missing_columns: frozenset[str] = frozenset()

    @property
    def is_clear(self) -> bool:
        return not self.missing_columns

    @property
    def message(self) -> str:
        sections = [
            f"⚠  GAP in {self.title!r} — missing MARIS columns: {sorted(self.missing_columns)}",
            "Easy Path (Check columns mapping): First, verify if simply adding raw provider column mappings to 'columns:' resolves this missing column.",
            "Hard Path (Boundary Loader): If the missing field is trapped inside workbook physics, mixed cells, combined coordinates, or value-unit strings, fix it in config/handlers/{dataset}_loader.py before the pipeline runs.",
        ]
        return "\n\n".join(part for part in sections if part)

    def raise_for_gap(self) -> None:
        if self.is_clear:
            return
        print(f"\n{self.message}\n")
        raise ValueError(f"YAML spec missing mappings for: {sorted(self.missing_columns)}")


def _gap_diagnostics(cfg: HandlerConfig) -> _GapDiagnostics:
    return _GapDiagnostics(
        title=cfg.title,
        missing_columns=cfg.missing_required_columns,
    )


def load_handler_config(cfg_cls, path: str | Path) -> HandlerConfig:
    raw = yaml.safe_load(Path(path).read_text(encoding="utf-8"))
    try:
        contract = _RawHandlerContract.model_validate(raw)
        unknown_attrs = set(contract.output.global_attrs) - NC_GLOBAL_ATTRS
        if unknown_attrs:
            msg = _unknown_global_attrs_message(unknown_attrs)
            print(msg)
            raise ValueError(msg)
        return cfg_cls(
            module_name=contract.handler.module_name,
            title=contract.handler.title,
            description=contract.handler.description.strip(),
            url=contract.data_source.url,
            fname_out=contract.data_source.fname_out,
            zenodo_id=contract.data_source.zenodo_id,
            fmt=contract.data_source.format,
            rename=contract.rename_cols.mapping,
            string_cast=contract.rename_cols.string_cast,
            columns=contract.columns,
            normalize_case=contract.normalize_case,
            col_date=contract.parse_datetime.col_date,
            col_time=contract.parse_datetime.col_time,
            dt_format=contract.parse_datetime.format,
            time_format=contract.time_format,
            meta_cols=contract.melt.meta_cols,
            melt_spec=contract.melt.spec,
            unit_conversions=contract.unit_conversions,
            nuclide_lut=contract.nomenclatures.nuclide_lut,
            unit_lut=contract.nomenclatures.unit_lut,
            lab_lut=contract.nomenclatures.lab_lut,
            lab_constants=contract.nomenclatures.lab_constants,
            area_default=contract.nomenclatures.area_default,
            keywords=contract.output.keywords,
            global_attrs=contract.output.global_attrs,
            loader=contract.loader,
        )
    except ValidationError as err:
        msg = _yaml_step1_message(err)
        print(msg)
        raise ValueError(msg) from err


def _stdout_supports(text: str) -> bool:
    encoding = getattr(sys.stdout, "encoding", None) or "utf-8"
    try:
        text.encode(encoding)
        return True
    except UnicodeEncodeError:
        return False


def _loader_suggestion_prefix() -> str:
    return "💡 SUGGESTION:" if _stdout_supports("💡") else "SUGGESTION:"


def _uses_default_loader(cfg: HandlerConfig) -> bool:
    if cfg.loader is None:
        return True
    return getattr(cfg.loader, "path", None) in _DEFAULT_LOADER_PATHS


def _loader_hint_name(cfg: HandlerConfig) -> str:
    stem = (cfg.module_name or "handler").split(".")[-1].strip() or "handler"
    return stem.replace("-", "_")


def _loader_read_hint(cfg: HandlerConfig) -> str:
    if cfg.fmt.lower() in {"xls", "xlsx", "excel"}:
        return "# df = pd.read_excel(cfg.url)"
    return "# df = pd.read_csv(cfg.url)"


def _custom_loader_skeleton(cfg: HandlerConfig, findings: list[dict[str, Any]]) -> str:
    grp = next((finding["grp"] for finding in findings if finding.get("grp")), "SEAWATER")
    lines = [
        _loader_suggestion_prefix() + " Do not patch this in the pipeline using callbacks.",
        'Use the "Boundary Ingestion Pattern" to cleanse this file before the pipeline runs.',
        "",
        f"[Step 1] Save this skeleton as: config/handlers/{_loader_hint_name(cfg)}_loader.py",
        "[Step 2] Point to it in your YAML under 'loader:'",
        "",
        "--- CUSTOM LOADER SKELETON ---",
        "import pandas as pd",
        "from marisco.handlers.pipeline.loader import HandlerConfig",
        "",
        "def load_and_cleanse(cfg: HandlerConfig) -> dict[str, pd.DataFrame]:",
        '    """',
        f"    Boundary Loader Skeleton for {cfg.title or cfg.module_name}.",
        "    TODO: Fix missing values in coordinates/time columns.",
        '    """',
        "    # 1. Load the raw file",
        f"    {_loader_read_hint(cfg)}  # or adapt to the provider format",
        "",
        "    # 2. Cleanse data (e.g., forward-fill, cast types)",
        "    # ...",
        "",
        "    # 3. Return a dictionary of cleansed DataFrames mapped to MARIS groups",
        f"    return {{'{grp}': df}}",
    ]
    return "\n".join(lines)


def _deep_gap_message(cfg: HandlerConfig, findings: list[dict[str, Any]]) -> str:
    sections = [
        "Gate 2 failed: the YAML contract declared canonical MARIS fields, but the loaded dataset is not semantically valid.",
        f"Dataset: {cfg.title or 'Untitled handler'}",
    ]
    skeleton_cols: set[str] = set()
    for finding in findings:
        lines = [f"Group '{finding['grp']}':"]
        if finding['missing']:
            lines.extend(f"- Missing required column: {col}." for col in finding['missing'])
            skeleton_cols.update(finding['missing'])
        if finding['null_counts']:
            lines.extend(f"- {col} contains {count} null value(s)." for col, count in finding['null_counts'].items())
            skeleton_cols.update(finding['null_counts'])
        if finding['empty_cols']:
            lines.extend(f"- {col} is present but empty in every row." for col in finding['empty_cols'])
            skeleton_cols.update(finding['empty_cols'])
        sections.append("\n".join(lines))
    sections.append("Encoding would be unsafe here because downstream time/coordinate rails can discard rows and mask the upstream defect.")
    sections.append(_custom_loader_skeleton(cfg, findings))
    if skeleton_cols and not _uses_default_loader(cfg):
        sections.append("Fix the configured boundary loader so the DataFrame is physically clean before declarative mapping begins.")
    return "\n\n".join(part for part in sections if part)


def gap_check(cfg: HandlerConfig, dfs: Optional[dict[str, pd.DataFrame]] = None) -> None:
    "Fail-Fast Gate 2: static contract quarantine first, deep data validation when dfs are provided."
    if dfs is None:
        _gap_diagnostics(cfg).raise_for_gap()
        return

    findings = []
    for grp, df in dfs.items():
        missing = sorted(_MARIS_REQUIRED - set(df.columns))
        null_counts = {
            col: int(df[col].isna().sum())
            for col in sorted(_DEEP_CRITICAL)
            if col in df.columns and int(df[col].isna().sum()) > 0
        }
        empty_cols = sorted(
            col for col in _MARIS_REQUIRED
            if col in df.columns and int(df[col].notna().sum()) == 0
        )
        if missing or null_counts or empty_cols:
            findings.append({
                'grp': grp,
                'missing': missing,
                'null_counts': null_counts,
                'empty_cols': empty_cols,
            })

    if not findings:
        return

    msg = _deep_gap_message(cfg, findings)
    print(f"\n{msg}\n")
    raise ValueError(msg)


In [ ]:
cfg_ok = HandlerConfig.from_yaml("config/handlers/fram_strait.yaml")
gap_check(cfg_ok)
print("gap_check(fram_strait) -> passed")